# 1. Generate Sampling Points

Generate flood and non-flood sample points from geospatial data for Tapanuli Tengah Regency.

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rasterio_mask
from shapely.geometry import Point, mapping
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

## 2. Configuration / 

In [ ]:
# Paths - adjust to your local environment
RASTER_PATH = r"data\elevasi_clip.tif" 
SHP_PATH = r"data\admin_tapteng_UTM47N.shp"
EXCEL_PATH = r"data\rekap_kecamatan.xlsx"

# Sampling parameters
TOTAL_SAMPLES = 600
FLOOD_RATIO = 0.5 # 50:50 flood vs non-flood

# Excel columns
COL_KEC_EXCEL = 'kecamatan'
COL_BANJIR_EXCEL = 'jumlah_banjir'

# Shapefile columns
COL_KEC_SHP = 'NAMOBJ'

# Pathlib Path for output directory
PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "samples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load Data / 

In [ ]:
df_excel = pd.read_excel(EXCEL_PATH)
gdf_shp = gpd.read_file(SHP_PATH)
src = rasterio.open(RASTER_PATH)

print(f'Excel: {len(df_excel)} affected sub-districts')
print(f'SHP: {len(gdf_shp)} total sub-districts')
print(f'Raster: {src.crs}, {src.width}x{src.height}')

# Reproject SHP if CRS differs
if gdf_shp.crs != src.crs:
    print(f'Reprojecting SHP {gdf_shp.crs} -> {src.crs}')
    gdf_shp = gdf_shp.to_crs(src.crs)

## 4. Validate Sub-district Names / 

In [ ]:
shp_kec = set(gdf_shp[COL_KEC_SHP].str.upper().str.strip())
excel_kec = set(df_excel[COL_KEC_EXCEL].str.upper().str.strip())

missing = excel_kec - shp_kec
if missing:
    print(f'Not found in SHP: {missing}')
else:
    print('All Excel sub-districts match SHP')

nonaffected = shp_kec - excel_kec
print(f'Affected: {len(excel_kec)}')
print(f'Non-affected: {len(nonaffected)}')

## 5. Sampling Function / 

Random points inside a polygon, ensuring they fall on valid raster pixels.

In [ ]:
def sample_random_in_polygon(polygon, raster_src, n_points):
    """Sample n_points random points inside a polygon on valid raster pixels."""
    try:
            out_image, out_transform = rasterio_mask(
                raster_src, [mapping(polygon)], crop=True, filled=False
            )
            data = out_image[0]
    
            # Pixel valid = tidak nodata
            rows, cols = np.where(~np.ma.getmaskarray(data))
    
            if len(rows) == 0:
                return []
    
            n_select = min(n_points, len(rows))
            chosen = np.random.choice(len(rows), n_select, replace=False)
    
            points = []
            for idx in chosen:
                r, c = rows[idx], cols[idx]
                x, y = rasterio.transform.xy(out_transform, r, c)
                points.append({
                    'geometry': Point(x, y),
                    'wo_value': int(data[r, c])
                })
    
            return points
    
    except Exception as e:
        print(f"    Error: {e}")
        return []

## 6. Proportional Allocation / 

In [ ]:
n_flood = int(TOTAL_SAMPLES * FLOOD_RATIO)
n_nonflood = TOTAL_SAMPLES - n_flood

total_kejadian = df_excel[COL_BANJIR_EXCEL].sum()
df_excel['alokasi'] = np.floor(
    (df_excel[COL_BANJIR_EXCEL] / total_kejadian) * n_flood
).astype(int)

sisa = n_flood - df_excel['alokasi'].sum()
if sisa > 0:
    df_excel.loc[df_excel[COL_BANJIR_EXCEL].idxmax(), 'alokasi'] += int(sisa)

print(f'Flood: {n_flood}, Non-flood: {n_nonflood}')
print(df_excel[[COL_KEC_EXCEL, COL_BANJIR_EXCEL, 'alokasi']].to_string(index=False))

## 7. Generate Flood Samples / 

In [ ]:
all_flood = []

for _, row in df_excel.iterrows():
    kec = row[COL_KEC_EXCEL]
    n_alloc = int(row['alokasi'])
    if n_alloc <= 0:
        continue

    match = gdf_shp[gdf_shp[COL_KEC_SHP].str.upper().str.strip() == kec.upper().strip()]
    if match.empty:
        print(f"  {kec}: not found in SHP")
        continue

    points = sample_random_in_polygon(match.geometry.values[0], src, n_alloc)
    for p in points:
        p['kecamatan'] = kec
        p['class'] = 1

    all_flood.extend(points)
    print(f"  {kec}: {len(points)}/{n_alloc}")

print(f"Total flood: {len(all_flood)}")

## 8. Generate Non-Flood Samples / 

In [ ]:
gdf_nonaffected = gdf_shp[
    ~gdf_shp[COL_KEC_SHP].str.upper().str.strip().isin(
        df_excel[COL_KEC_EXCEL].str.upper().str.strip()
    )
]

if len(gdf_nonaffected) == 0:
    print("No non-affected sub-districts, sampling from entire area")
    gdf_nonaffected = gdf_shp.copy()

n_per_kec = max(1, n_nonflood // len(gdf_nonaffected))
all_nonflood = []
remaining = n_nonflood

print(f"Non-affected sub-districts: {len(gdf_nonaffected)}, target ~{n_per_kec}/sub-district")

for _, row in gdf_nonaffected.iterrows():
    if remaining <= 0:
        break

    kec = row[COL_KEC_SHP]
    n_alloc = min(n_per_kec, remaining)

    points = sample_random_in_polygon(row.geometry, src, n_alloc)
    for p in points:
        p['kecamatan'] = kec
        p['class'] = 0

    all_nonflood.extend(points)
    remaining -= len(points)
    print(f"  {kec}: {len(points)}")

if remaining > 0:
    print(f"Remaining {remaining} points, sampling from entire area")
    extra = sample_random_in_polygon(gdf_shp.unary_union, src, remaining)
    for p in extra:
        p['kecamatan'] = 'lainnya'
        p['class'] = 0
    all_nonflood.extend(extra)

print(f"Total non-banjir: {len(all_nonflood)}")

## 9. Export Combined Sample / 

In [ ]:
all_points = all_flood + all_nonflood

if len(all_points) == 0:
    print("ERROR: Tidak ada titik")
else:
    gdf = gpd.GeoDataFrame(all_points, crs=gdf_shp.crs)
    gdf['lon'] = gdf.geometry.x
    gdf['lat'] = gdf.geometry.y
    gdf = gdf[['class', 'kecamatan', 'wo_value', 'lon', 'lat', 'geometry']]

# Save as CSV for downstream notebooks
OUTPUT_FILE = OUTPUT_DIR / "sample_points.csv"
df_out = gdf[['class', 'kecamatan', 'wo_value', 'lat', 'lon']].copy()
df_out.rename(columns={'class': 'label'}, inplace=True)
df_out.to_csv(OUTPUT_FILE, index=False)
print(f'Saved {len(df_out)} points to {OUTPUT_FILE}')

src.close()

## 10. Visualization / 

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
gdf_shp.boundary.plot(ax=ax, color='gray', linewidth=0.5)
gdf[gdf['class'] == 1].plot(ax=ax, color='red', markersize=15, label='Flood', alpha=0.7)
gdf[gdf['class'] == 0].plot(ax=ax, color='blue', markersize=15, label='Non-Flood', alpha=0.7)
ax.set_title('Flood & Non-Flood Sample Points', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()